# W2 D1 - Alert Correlator

Pipeline: Dedup -> Session Window -> Topology Grouping -> `results/cluster_summary.json`.

In [1]:
import json
from collections import Counter, defaultdict, deque
from datetime import datetime
from pathlib import Path

BASE = Path.cwd()
DATASET = BASE / 'dataset' / 'alerts_sample.jsonl'
SERVICES = BASE / 'dataset' / 'services.json'
OUT = BASE / 'results' / 'cluster_summary.json'

GAP_SEC = 120
MAX_HOP = 2
SEVERITY_RANK = {'info': 0, 'warn': 1, 'warning': 1, 'crit': 2, 'critical': 2}

def parse_ts(value):
    return datetime.fromisoformat(value.replace('Z', '+00:00'))

def format_ts(value):
    return value.isoformat().replace('+00:00', 'Z')

def load_alerts(path):
    alerts = []
    with path.open(encoding='utf-8') as f:
        for line in f:
            if not line.strip():
                continue
            alert = json.loads(line)
            alert['timestamp'] = parse_ts(alert.get('timestamp') or alert['ts'])
            alert['fingerprint'] = fingerprint(alert)
            alerts.append(alert)
    return sorted(alerts, key=lambda x: x['timestamp'])

def load_graph(path):
    graph_data = json.loads(path.read_text(encoding='utf-8'))
    graph = defaultdict(set)
    stores = {store['name'] for store in graph_data.get('stores', [])}
    for svc in graph_data.get('services', []):
        graph[svc['name'] if isinstance(svc, dict) else svc]
    for store in stores:
        graph[store]
    for edge in graph_data['edges']:
        src, dst = (edge['from'], edge['to']) if isinstance(edge, dict) else edge
        graph[src].add(dst)
        graph[dst].add(src)
    return graph, stores

def fingerprint(alert):
    return f"{alert['service']}|{alert['metric']}|{alert['severity']}"

def dedup(alerts):
    counts = Counter(alert['fingerprint'] for alert in alerts)
    representatives = {}
    for alert in alerts:
        representatives.setdefault(alert['fingerprint'], alert)
    return [dict(alert, duplicate_count=counts[fp]) for fp, alert in representatives.items()], counts

def session_groups(alerts, gap_sec):
    sessions = []
    current = []
    last_ts = None
    for alert in alerts:
        if last_ts is None or (alert['timestamp'] - last_ts).total_seconds() <= gap_sec:
            current.append(alert)
        else:
            sessions.append(current)
            current = [alert]
        last_ts = alert['timestamp']
    if current:
        sessions.append(current)
    return sessions

def within_hops(graph, start, max_hop, stores, alerted_services):
    seen = {start}
    q = deque([(start, 0)])
    while q:
        node, depth = q.popleft()
        if depth == max_hop:
            continue
        if node in stores and node not in alerted_services and node != start:
            continue
        for nxt in graph[node]:
            if nxt not in seen:
                seen.add(nxt)
                q.append((nxt, depth + 1))
    return seen

def topology_group(session, graph, stores, max_hop):
    unassigned = list(session)
    alerted_services = {alert['service'] for alert in session}
    groups = []
    while unassigned:
        seed = max(
            unassigned,
            key=lambda alert: (SEVERITY_RANK.get(alert['severity'], -1), -alert['timestamp'].timestamp()),
        )
        reachable = within_hops(graph, seed['service'], max_hop, stores, alerted_services)
        group = [alert for alert in unassigned if alert['service'] in reachable]
        grouped_ids = {alert['id'] for alert in group}
        groups.append(group)
        unassigned = [alert for alert in unassigned if alert['id'] not in grouped_ids]
    return sorted(groups, key=lambda group: min(alert['timestamp'] for alert in group))

def build_cluster(cluster_index, session_index, alerts):
    timestamps = [alert['timestamp'] for alert in alerts]
    severities = [alert['severity'] for alert in alerts]
    max_severity = max(severities, key=lambda sev: SEVERITY_RANK.get(sev, -1))
    return {
        'cluster_id': f'c-{session_index:03d}-{cluster_index:03d}',
        'alert_count': len(alerts),
        'services': sorted({alert['service'] for alert in alerts}),
        'time_range': [format_ts(min(timestamps)), format_ts(max(timestamps))],
        'max_severity': max_severity,
        'fingerprints': sorted({alert['fingerprint'] for alert in alerts}),
    }

def correlate():
    alerts = load_alerts(DATASET)
    graph, stores = load_graph(SERVICES)
    _, duplicate_counts = dedup(alerts)
    sessions = session_groups(alerts, GAP_SEC)

    clusters = []
    for session_index, session in enumerate(sessions, start=1):
        topology_groups = topology_group(session, graph, stores, MAX_HOP)
        for cluster_index, group in enumerate(topology_groups):
            clusters.append(build_cluster(cluster_index, session_index, group))

    summary = {
        'input_alerts': len(alerts),
        'output_clusters': len(clusters),
        'reduction_ratio': round(1 - len(clusters) / len(alerts), 2),
        'clusters': clusters,
    }
    OUT.parent.mkdir(parents=True, exist_ok=True)
    OUT.write_text(json.dumps(summary, indent=2), encoding='utf-8')
    return summary

summary = correlate()
summary

{'input_alerts': 20,
 'output_clusters': 3,
 'reduction_ratio': 0.85,
 'clusters': [{'cluster_id': 'c-001-000',
   'alert_count': 18,
   'services': ['cart-svc',
    'checkout-svc',
    'edge-lb',
    'notification-svc',
    'payment-svc'],
   'time_range': ['2026-06-12T09:42:01Z', '2026-06-12T09:48:30Z'],
   'max_severity': 'crit',
   'fingerprints': ['cart-svc|latency_p99_ms|warn',
    'checkout-svc|downstream_payment_error_rate|crit',
    'checkout-svc|latency_p99_ms|crit',
    'checkout-svc|latency_p99_ms|warn',
    'checkout-svc|request_drop_rate|crit',
    'edge-lb|p99_latency_ms|crit',
    'edge-lb|upstream_5xx_rate|crit',
    'edge-lb|upstream_5xx_rate|warn',
    'notification-svc|queue_depth|crit',
    'notification-svc|queue_lag_ms|warn',
    'payment-svc|db_connection_pool_used_ratio|crit',
    'payment-svc|db_connection_pool_used_ratio|warn',
    'payment-svc|error_rate|crit',
    'payment-svc|error_rate|warn',
    'payment-svc|latency_p99_ms|crit']},
  {'cluster_id': 'c-00